
# Manufacturing Performance Analysis

## Project Overview

This project analyses manufacturing production and quality data
to identify performance gaps across machines and departments.

The objective is to transform raw manufacturing data into
meaningful insights that support operational decision-making.

## Business Objectives

- Measure production achievement against targets.
- Compare performance across departments and machines.
- Examine manufacturing defect rates.
- Investigate downtime and maintenance patterns.

## Tools and Technologies

- Python
- pandas
- Google Colab
- SQLite
- Microsoft Excel
- Power BI


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Manufacturing_Portfolio_Data.xlsx to Manufacturing_Portfolio_Data.xlsx


In [ ]:
import pandas as pd

production = pd.read_excel(
    "Manufacturing_Portfolio_Data.xlsx",
    sheet_name="Production",
)

production.head()

,RecordID,ProductionDate,MachineID,Shift,TargetUnits,ProductionUnits,DefectUnits,ScheduledMinutes,DowntimeMinutes
0,P-0001,2026-07-01,M-101,Day,650,618,7,480,38
1,P-0002,2026-07-01,M-101,Night,650,597,14,480,34
2,P-0003,2026-07-01,M-102,Day,650,585,9,480,72
3,P-0004,2026-07-01,M-102,Night,650,601,7,480,54
4,P-0005,2026-07-01,M-103,Day,750,746,14,480,15


In [ ]:
production.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 722 entries, 0 to 721
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   RecordID          722 non-null    object        
 1   ProductionDate    722 non-null    datetime64[ns]
 2   MachineID         722 non-null    object        
 3   Shift             722 non-null    object        
 4   TargetUnits       722 non-null    int64         
 5   ProductionUnits   722 non-null    int64         
 6   DefectUnits       722 non-null    int64         
 7   ScheduledMinutes  722 non-null    int64         
 8   DowntimeMinutes   722 non-null    int64         
dtypes: datetime64[ns](1), int64(5), object(3)
memory usage: 50.9+ KB


In [ ]:
production.isna().sum()

,0
RecordID,0
ProductionDate,0
MachineID,0
Shift,0
TargetUnits,0
ProductionUnits,0
DefectUnits,0
ScheduledMinutes,0
DowntimeMinutes,0


In [ ]:
production["ProductionDate"] = pd.to_datetime(production["ProductionDate"])

In [ ]:
production.duplicated().sum()

np.int64(2)

In [ ]:
production_clean = production.drop_duplicates().copy()

production_clean.shape

(720, 9)

In [ ]:
production_clean.duplicated().sum()

np.int64(0)

In [ ]:
production_clean["Shift"].unique()

array(['Day', 'Night', ' Night '], dtype=object)

In [ ]:
production_clean["Shift"] = production_clean["Shift"].str.strip()

In [ ]:
production_clean["Shift"].unique()

array(['Day', 'Night'], dtype=object)

In [ ]:
production_clean["Shift"].value_counts()

,count
Shift,
Day,360
Night,360


In [ ]:
production_clean["GoodUnits"] = (
    production_clean["ProductionUnits"]
    - production_clean["DefectUnits"]
)

In [ ]:
production_clean[
    ["ProductionUnits", "DefectUnits", "GoodUnits"]
].head()

,ProductionUnits,DefectUnits,GoodUnits
0,618,7,611
1,597,14,583
2,585,9,576
3,601,7,594
4,746,14,732


In [ ]:
production_clean["GoodUnits"].sum()

np.int64(433992)

In [ ]:
production_clean.shape

(720, 10)

In [ ]:
below_target = production_clean[production_clean["ProductionUnits"]
                                <production_clean["TargetUnits"]].copy()
below_target.head()

,RecordID,ProductionDate,MachineID,Shift,TargetUnits,ProductionUnits,DefectUnits,ScheduledMinutes,DowntimeMinutes,GoodUnits
0,P-0001,2026-07-01,M-101,Day,650,618,7,480,38,611
1,P-0002,2026-07-01,M-101,Night,650,597,14,480,34,583
2,P-0003,2026-07-01,M-102,Day,650,585,9,480,72,576
3,P-0004,2026-07-01,M-102,Night,650,601,7,480,54,594
4,P-0005,2026-07-01,M-103,Day,750,746,14,480,15,732


In [ ]:
below_target.shape

(649, 10)

In [ ]:
below_target["ShortfallUnits"] = (below_target["TargetUnits"] - below_target["ProductionUnits"])

In [ ]:
below_target[["MachineID", "ProductionUnits", "TargetUnits", "ShortfallUnits"]].head()


,MachineID,ProductionUnits,TargetUnits,ShortfallUnits
0,M-101,618,650,32
1,M-101,597,650,53
2,M-102,585,650,65
3,M-102,601,650,49
4,M-103,746,750,4


In [ ]:
below_target.shape

(649, 11)

In [ ]:
largest_shortfalls = below_target.sort_values(
    by = "ShortfallUnits",
    ascending=False
)


largest_shortfalls[
    [
        "ProductionDate",
        "MachineID",
        "Shift",
        "ProductionUnits",
        "TargetUnits",
        "ShortfallUnits"
    ]
].head(5)

,ProductionDate,MachineID,Shift,ProductionUnits,TargetUnits,ShortfallUnits
78,2026-07-10,M-104,Day,471,600,129
135,2026-07-17,M-104,Night,474,600,126
93,2026-07-12,M-103,Night,628,750,122
559,2026-09-08,M-104,Night,484,600,116
181,2026-07-23,M-103,Night,634,750,116


In [ ]:
machine_production = production_clean.groupby("MachineID")["ProductionUnits"].sum()
machine_production

,ProductionUnits
MachineID,
M-101,110058
M-102,110331
M-103,126601
M-104,96387


In [ ]:
machine_production.sum()

np.int64(443377)

In [ ]:
machine_summary = production_clean.groupby("MachineID",
as_index = False)[["ProductionUnits", "TargetUnits", "DefectUnits", "GoodUnits"]].sum()
machine_summary

,MachineID,ProductionUnits,TargetUnits,DefectUnits,GoodUnits
0,M-101,110058,117000,1990,108068
1,M-102,110331,117000,1966,108365
2,M-103,126601,135000,2313,124288
3,M-104,96387,108000,3116,93271


In [ ]:
machine_summary["TargetUnits"].sum()

np.int64(477000)

In [ ]:
machine_summary["AchievementPct"] = (
    machine_summary["ProductionUnits"] / machine_summary["TargetUnits"]
    *100
)

machine_summary["DefectPct"] = (
    machine_summary["DefectUnits"]/machine_summary["ProductionUnits"]
    *100
)

machine_summary.round(2)


,MachineID,ProductionUnits,TargetUnits,DefectUnits,GoodUnits,AchievementPct,DefectPct
0,M-101,110058,117000,1990,108068,94.07,1.81
1,M-102,110331,117000,1966,108365,94.30,1.78
2,M-103,126601,135000,2313,124288,93.78,1.83
3,M-104,96387,108000,3116,93271,89.25,3.23


In [ ]:
machines = pd.read_excel("Manufacturing_Portfolio_Data.xlsx",
                         sheet_name="Machines")
machines.head()

,MachineID,MachineName,Department
0,M-101,CNC Lathe 1,Machining
1,M-102,CNC Lathe 2,Machining
2,M-103,Assembly Line 1,Assembly
3,M-104,Assembly Line 2,Assembly


In [ ]:
machine_report = machine_summary.merge(
    machines,
    on="MachineID",
    how="left",
    validate="one_to_one"
)

machine_report.round(2)

,MachineID,ProductionUnits,TargetUnits,DefectUnits,GoodUnits,AchievementPct,DefectPct,MachineName,Department
0,M-101,110058,117000,1990,108068,94.07,1.81,CNC Lathe 1,Machining
1,M-102,110331,117000,1966,108365,94.30,1.78,CNC Lathe 2,Machining
2,M-103,126601,135000,2313,124288,93.78,1.83,Assembly Line 1,Assembly
3,M-104,96387,108000,3116,93271,89.25,3.23,Assembly Line 2,Assembly


In [ ]:
machine_report[
    ["MachineID", "MachineName", "Department", "AchievementPct"]
].round(2)

,MachineID,MachineName,Department,AchievementPct
0,M-101,CNC Lathe 1,Machining,94.07
1,M-102,CNC Lathe 2,Machining,94.30
2,M-103,Assembly Line 1,Assembly,93.78
3,M-104,Assembly Line 2,Assembly,89.25



## Key Findings

### Department Performance Comparison

- Machining achieved 94.18% of its production target.
- Assembly achieved 91.76% of its production target.
- Machining's production achievement was 2.42 percentage points
  higher than Assembly's.
- Machining recorded a defect rate of 1.80%, compared with
  2.43% for Assembly.

### Business Interpretation

In the analysed data, Assembly had lower production target
achievement and a higher defect rate than Machining.

This suggests that Assembly could be prioritised for further
investigation into production constraints and quality issues.

These results show a performance difference but do not establish
the underlying causes.

### Next Analysis

Investigate machine-level downtime and maintenance records
to identify potential opportunities for operational improvement.


In [ ]:
department_summary = machine_report.groupby(
    "Department",
    as_index=False
)[
    ["ProductionUnits", "TargetUnits", "DefectUnits", "GoodUnits"]
].sum()

department_summary

,Department,ProductionUnits,TargetUnits,DefectUnits,GoodUnits
0,Assembly,222988,243000,5429,217559
1,Machining,220389,234000,3956,216433


In [ ]:
department_summary["AchievementPct"] = (
    department_summary["ProductionUnits"]
    / department_summary["TargetUnits"]
    * 100
)

department_summary["DefectPct"] = (
    department_summary["DefectUnits"]
    / department_summary["ProductionUnits"]
    * 100
)

department_summary.round(2)

,Department,ProductionUnits,TargetUnits,DefectUnits,GoodUnits,AchievementPct,DefectPct
0,Assembly,222988,243000,5429,217559,91.76,2.43
1,Machining,220389,234000,3956,216433,94.18,1.80


In [ ]:
department_ranked = department_summary.sort_values(
    by="AchievementPct",
    ascending=False
)

department_ranked[["Department", "AchievementPct", "DefectPct"]].round(2)

,Department,AchievementPct,DefectPct
1,Machining,94.18,1.80
0,Assembly,91.76,2.43


In [ ]:
machine_downtime = production_clean.groupby(
    "MachineID",
    as_index=False
)[["DowntimeMinutes","ScheduledMinutes"]].sum()

machine_downtime.head()


,MachineID,DowntimeMinutes,ScheduledMinutes
0,M-101,7608,86400
1,M-102,7436,86400
2,M-103,7869,86400
3,M-104,12888,86400


In [ ]:
machine_downtime["DowntimePct"] = (
    machine_downtime["DowntimeMinutes"]
    / machine_downtime["ScheduledMinutes"]
    * 100
)

machine_downtime.sort_values(by="DowntimePct", ascending=False).round(2)

,MachineID,DowntimeMinutes,ScheduledMinutes,DowntimePct
3,M-104,12888,86400,14.92
2,M-103,7869,86400,9.11
0,M-101,7608,86400,8.81
1,M-102,7436,86400,8.61


In [ ]:
machine_performance = machine_report.merge(
    machine_downtime,
    on="MachineID",
    how="left",
    validate="one_to_one"
)
machine_performance[
    [
        "MachineID",
        "MachineName",
        "AchievementPct",
        "DefectPct",
        "DowntimePct"
    ]
].round(2)

,MachineID,MachineName,AchievementPct,DefectPct,DowntimePct
0,M-101,CNC Lathe 1,94.07,1.81,8.81
1,M-102,CNC Lathe 2,94.30,1.78,8.61
2,M-103,Assembly Line 1,93.78,1.83,9.11
3,M-104,Assembly Line 2,89.25,3.23,14.92


In [ ]:
machine_performance[
    machine_performance["MachineID"] == "M-104"
].round(2)

,MachineID,ProductionUnits,TargetUnits,DefectUnits,GoodUnits,AchievementPct,DefectPct,MachineName,Department,DowntimeMinutes,ScheduledMinutes,DowntimePct
3,M-104,96387,108000,3116,93271,89.25,3.23,Assembly Line 2,Assembly,12888,86400,14.92


In [ ]:
production_clean["ProductionMonth"] = production_clean["ProductionDate"].dt.strftime("%Y-%m")
production_clean[["ProductionDate", "ProductionMonth"]].head()

,ProductionDate,ProductionMonth
0,2026-07-01,2026-07
1,2026-07-01,2026-07
2,2026-07-01,2026-07
3,2026-07-01,2026-07
4,2026-07-01,2026-07


In [ ]:
production_clean["ProductionMonth"].unique()

array(['2026-07', '2026-08', '2026-09'], dtype=object)

In [ ]:
monthly_summary = production_clean.groupby("ProductionMonth",
                                           as_index=False)[
    [
        "ProductionUnits",
        "TargetUnits",
        "DefectUnits",
        "GoodUnits"
    ]
].sum()

monthly_summary

,ProductionMonth,ProductionUnits,TargetUnits,DefectUnits,GoodUnits
0,2026-07,152364,164300,3314,149050
1,2026-08,152886,164300,3256,149630
2,2026-09,138127,148400,2815,135312


In [ ]:
monthly_summary["AchievementPct"] = monthly_summary["ProductionUnits"]/monthly_summary["TargetUnits"]*100
monthly_summary.round(2)

,ProductionMonth,ProductionUnits,TargetUnits,DefectUnits,GoodUnits,AchievementPct
0,2026-07,152364,164300,3314,149050,92.74
1,2026-08,152886,164300,3256,149630,93.05
2,2026-09,138127,148400,2815,135312,93.08


In [ ]:
monthly_summary["DefectPct"] = (
    monthly_summary["DefectUnits"]
    / monthly_summary["ProductionUnits"]
    * 100
)

In [ ]:
monthly_downtime = production_clean.groupby("ProductionMonth",
                                           as_index=False)[["DowntimeMinutes", "ScheduledMinutes"]].sum()
monthly_downtime

,ProductionMonth,DowntimeMinutes,ScheduledMinutes
0,2026-07,12679,119040
1,2026-08,11895,119040
2,2026-09,11227,107520


In [ ]:
monthly_downtime["DowntimePct"] = (
    monthly_downtime["DowntimeMinutes"]
    / monthly_downtime["ScheduledMinutes"]
    * 100
)

monthly_performance = monthly_summary.merge(
    monthly_downtime,
    on="ProductionMonth",
    how="left",
    validate="one_to_one"
)

monthly_performance[
    [
        "ProductionMonth",
        "AchievementPct",
        "DefectPct",
        "DowntimePct"
    ]
].round(2)

,ProductionMonth,AchievementPct,DefectPct,DowntimePct
0,2026-07,92.74,2.18,10.65
1,2026-08,93.05,2.13,9.99
2,2026-09,93.08,2.04,10.44


In [ ]:
monthly_performance.sort_values(by="DowntimePct", ascending=False)
monthly_performance[["ProductionMonth", "DefectPct"]
].head(1).round(2)

,ProductionMonth,DefectPct
0,2026-07,2.18


In [ ]:
monthly_trend = monthly_performance.sort_values(by="ProductionMonth").copy()

In [ ]:
monthly_trend["AchievementChangePP"] = (
    monthly_trend["AchievementPct"].diff()
)

monthly_trend[
    [
        "ProductionMonth",
        "AchievementPct",
        "AchievementChangePP"
    ]
].round(2)

,ProductionMonth,AchievementPct,AchievementChangePP
0,2026-07,92.74,NaN
1,2026-08,93.05,0.32
2,2026-09,93.08,0.02


In [ ]:
shift_summary = production_clean.groupby(
    "Shift",
    as_index=False
)[["ProductionUnits",
  "TargetUnits",
  "DefectUnits",
  "ScheduledMinutes",
  "DowntimeMinutes"
  ]
 ].sum()

shift_summary

,Shift,ProductionUnits,TargetUnits,DefectUnits,ScheduledMinutes,DowntimeMinutes
0,Day,221064,238500,4541,172800,18130
1,Night,222313,238500,4844,172800,17671


In [ ]:
shift_summary["AchievementPct"] = shift_summary["ProductionUnits"]/shift_summary["TargetUnits"]*100
shift_summary["DefectPct"] = shift_summary["DefectUnits"]/shift_summary["ProductionUnits"]*100
shift_summary["DowntimePct"] = shift_summary["DowntimeMinutes"]/shift_summary["ScheduledMinutes"]*100
shift_summary[
    ["Shift", "AchievementPct", "DefectPct", "DowntimePct"]
].round(2)

,Shift,AchievementPct,DefectPct,DowntimePct
0,Day,92.69,2.05,10.49
1,Night,93.21,2.18,10.23


In [ ]:
production_clean.head(20)

,RecordID,ProductionDate,MachineID,Shift,TargetUnits,ProductionUnits,DefectUnits,ScheduledMinutes,DowntimeMinutes,GoodUnits,ProductionMonth
0,P-0001,2026-07-01,M-101,Day,650,618,7,480,38,611,2026-07
1,P-0002,2026-07-01,M-101,Night,650,597,14,480,34,583,2026-07
2,P-0003,2026-07-01,M-102,Day,650,585,9,480,72,576,2026-07
3,P-0004,2026-07-01,M-102,Night,650,601,7,480,54,594,2026-07
4,P-0005,2026-07-01,M-103,Day,750,746,14,480,15,732,2026-07
5,P-0006,2026-07-01,M-103,Night,750,740,19,480,26,721,2026-07
6,P-0007,2026-07-01,M-104,Day,600,576,24,480,41,552,2026-07
7,P-0008,2026-07-01,M-104,Night,600,585,24,480,60,561,2026-07
8,P-0009,2026-07-02,M-101,Day,650,564,5,480,49,559,2026-07
9,P-0010,2026-07-02,M-101,Night,650,608,10,480,16,598,2026-07


In [ ]:
production_clean.to_csv("production_clean.csv", index=False, date_format="%Y-%m-%d")

In [ ]:
export_check = pd.read_csv("production_clean.csv")
print(export_check.shape)

(720, 11)


In [ ]:
from google.colab import files
files.download("production_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
with pd.ExcelWriter("manufacturing_summary_report.xlsx") as writer:
  machine_performance.to_excel(
      writer,
      sheet_name="Machine Performance",
      index=False
  )
  monthly_performance.to_excel(
      writer,
      sheet_name="MonthlyPerformance",
      index=False
  )
  monthly_trend.to_excel(
      writer,
      sheet_name="MonthlyTrend",
      index=False
  )
  shift_summary.to_excel(
      writer,
      sheet_name="ShiftSummary",
      index=False
  )

In [ ]:
from google.colab import files
files.download("manufacturing_summary_report.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>